In [1]:
import ee
import os
import geemap
from dotenv import load_dotenv
load_dotenv()

# Initialize the Earth Engine module.
PROJECT_ID = os.getenv("PROJECT_ID")
ee.Initialize(project = os.getenv("PROJECT_ID"))

# Define Denmark Area
DENMARK = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017').filter(ee.Filter.eq('country_na', 'Denmark')).geometry()

# Define map visual parameters
DW_CLASS = {
    'water' : '419bdf', # blue
    'trees' : '397d49', # green
    'grass' : '88b053', # light green
    'flooded_vegetation' : '7a87c6', # greyish blue
    'crops' : 'e49635', # orange
    'shrub_and_scrub' : 'dfc35a', # yellow
    'built' : 'c4281b', # red
    'bare' : 'a59b8f', # grey
    'snow_and_ice' : 'b39fe1', # purple
}

DW_VIZ_PARAMS = {
  'min' : 0,
  'max' : 8,
  'palette' : list(DW_CLASS.values()),
  'bands' : 'label'
}

S2_VIZ_PARAMS = {
  'min' : 0,
  'max' : 3000,
  'bands' : ['B4', 'B3', 'B2']
}

# Calculate area of an image in km²
def dw_area(img):
    area_image = img.multiply(ee.Image.pixelArea())
    area = area_image.reduceRegion( 
        reducer = ee.Reducer.sum(),
        scale = 10,
        maxPixels = 1e13,
    )
    return ee.Number(area.get('label')).divide(pow(10,6))

# Calculate change matrix
# 9x9 matrix area classified per class vs area classified per class other year
def change_matrix(img, img_prev):
    for i in range(9):
        for j in range(9):
            dw_transition = img.eq(i).And(img_prev.eq(j))
            dw_transition_area = dw_area(dw_transition)
            display(dw_transition_area)

# Calculate how many changes happen in a collection of images per pixel
def count_changes_collection(collection):
    img = collection.sum()

    counts = img.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram().unweighted(),
        geometry=DENMARK,
        scale=10,         
        maxPixels=1e13       
    )

    print(counts.getInfo())

# Return the amount of pixels
def count_pixels(img):
    return img.reduceRegion(
        reducer=ee.Reducer.sum().unweighted(),
        geometry=DENMARK,
        scale=10,
        maxPixels=1e13
    )



In [ ]:
YEAR = 25

# DANISH AGRICULTURE AGENCY
dataset = ee.FeatureCollection("projects/" + PROJECT_ID + "/assets/Markblokke" + str(YEAR))
dataset_prev = ee.FeatureCollection("projects/" + PROJECT_ID + "/assets/Markblokke" + str(YEAR - 1))
# Transform data to raster format for lighter operations
daa_img = ee.Image.constant(0).paint(dataset, 1)
daa_img_prev = ee.Image.constant(0).paint(dataset_prev, 1)
# Image with are that changed from DAA (1 - new field, -1 - removed field)
daa_change_img = daa_img.subtract(daa_img_prev)
daa_change_img = daa_change_img.updateMask(daa_change_img.neq(0))


# DYNAMIC WORLD CLASSIFICATION OF DAA AREA
dw_img = ee.Image('projects/' + PROJECT_ID + '/assets/dw_20' + str(YEAR) + '_mode').clipToCollection(dataset)
dw_img_prev = ee.Image('projects/' + PROJECT_ID + '/assets/dw_20' + str(YEAR - 1) + '_mode').clipToCollection(dataset_prev)

## Change Matrix

How did classifications change from year to year? What classes stay the same? Which ones turned into something different?

In [ ]:
change_matrix(dw_img, dw_img_prev)

#### New Fields Area

From the new field area, how did classification change? How was it classified before? How is it classified now?

In [ ]:
change_matrix(dw_img.And(daa_change_img.eq(1)), dw_img_prev.And(daa_change_img.eq(1)))

#### Removed Fields Area

From the removed field area, how did classification change? How was it classified before? How is it classified now?

In [ ]:
change_matrix(dw_img.And(daa_change_img.eq(-1)), dw_img_prev.And(daa_change_img.eq(-1)))

## Amount of Changes

In [13]:
# DAA Datasets from 2020-2025
dataset_25 = ee.FeatureCollection("projects/" + PROJECT_ID + "/assets/Markblokke25")
dataset_24 = ee.FeatureCollection("projects/" + PROJECT_ID + "/assets/Markblokke24")
dataset_23 = ee.FeatureCollection("projects/" + PROJECT_ID + "/assets/Markblokke23")
dataset_22 = ee.FeatureCollection("projects/" + PROJECT_ID + "/assets/Markblokke22")
dataset_21 = ee.FeatureCollection("projects/" + PROJECT_ID + "/assets/Markblokke21")
dataset_20 = ee.FeatureCollection("projects/" + PROJECT_ID + "/assets/Markblokke21")

# DW Images from 2020-2025 cropped to DAA area
dw_img_25 = ee.Image('projects/' + PROJECT_ID + '/assets/dw_2025_mode').clipToCollection(dataset_25)
dw_img_24 = ee.Image('projects/' + PROJECT_ID + '/assets/dw_2024_mode').clipToCollection(dataset_24)
dw_img_23 = ee.Image('projects/' + PROJECT_ID + '/assets/dw_2023_mode').clipToCollection(dataset_23)
dw_img_22 = ee.Image('projects/' + PROJECT_ID + '/assets/dw_2022_mode').clipToCollection(dataset_22)
dw_img_21 = ee.Image('projects/' + PROJECT_ID + '/assets/dw_2021_mode').clipToCollection(dataset_21)
dw_img_20 = ee.Image('projects/' + PROJECT_ID + '/assets/dw_2020_mode').clipToCollection(dataset_20)

# Yearly Change Images
dw_img_change_25_24 = dw_img_25.neq(dw_img_24)
dw_img_change_24_23 = dw_img_24.neq(dw_img_23)
dw_img_change_23_22 = dw_img_23.neq(dw_img_22)
dw_img_change_22_21 = dw_img_22.neq(dw_img_21)
dw_img_change_21_20 = dw_img_21.neq(dw_img_20)

# Create one image with all the classification from 20-25
dw_img_20_25 = ee.Image().addBands(dw_img_20)
dw_img_20_25 = dw_img_20_25.addBands(dw_img_21)
dw_img_20_25 = dw_img_20_25.addBands(dw_img_22)
dw_img_20_25 = dw_img_20_25.addBands(dw_img_23)
dw_img_20_25  = dw_img_20_25.addBands(dw_img_24)
dw_img_20_25 = dw_img_20_25.addBands(dw_img_25)

change_collection = ee.ImageCollection.fromImages(
    [dw_img_change_25_24, dw_img_change_24_23, dw_img_change_23_22, dw_img_change_22_21, dw_img_change_21_20]
)

In [ ]:
pixel_count = count_pixels(dw_img_change_25_24)
print('This many classifications changed from 2024 to 2025:', pixel_count.get('label').getInfo())

pixel_count = count_pixels(dw_img_change_24_23)
print('This many classifications changed from 2023 to 2024:', pixel_count.get('label').getInfo())

pixel_count = count_pixels(dw_img_change_23_22)
print('This many classifications changed from 2022 to 2023:', pixel_count.get('label').getInfo())

pixel_count = count_pixels(dw_img_change_22_21)
print('This many classifications changed from 2021 to 2022:', pixel_count.get('label').getInfo())

pixel_count = count_pixels(dw_img_change_21_20)
print('This many classifications changed from 2021 to 2020:', pixel_count.get('label').getInfo())

In [ ]:
display(count_pixels(dw_img_change_25_24.updateMask(change_collection.sum().eq(1))))

In [19]:
display(count_pixels(dw_img_change_25_24.updateMask(change_collection.sum().eq(1))))
print(30601172/59249438*100)
print(30601172/83265438*100)

51.6480375729471
36.751349341367785


In [4]:
count_changes_collection(change_collection)

{'label': {'0': 326198553, '1': 59249438, '2': 79957911, '3': 37661388, '4': 12112201, '5': 1778025, 'null': 250434972}}


### Visual Representation of Areas with More Change

In [8]:
always_change = dw_img_20_25.updateMask(change_collection.sum().eq(5))

map = geemap.Map(scroll_wheel_zoom = False, basemap='ROADMAP')
map.add_layer(
    always_change,
    DW_VIZ_PARAMS,
    'Dynamic World',
)
map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

### Rapid Sequence Change

From all the changes between two years, this many got reverted the year later

|           | Total No. Change | Changes Reverted Year After | Percentage |
|-----------|------------------|----------|--------|
| 2024-2025 |         85806833 |          |       |
| 2023-2024 |         62425362 | 22893903 | 36.67 |
| 2022-2023 |         85195804 | 21401276 | 25.12 |
| 2021-2022 |        88837093  | 47183936 | 53.11 |
| 2020-2021 |         67223154 | 28416582 | 42.27 |


In [7]:
rsc_20_21 = dw_img_20_25.select('label').neq(dw_img_20_25.select('label_1')) \
    .And(dw_img_20_25.select('label').eq(dw_img_20_25.select('label_2')))

pixel_count = count_pixels(rsc_20_21)
print('Changes from 2020 to 2021 that got reverted 2022:', pixel_count.get('label').getInfo())

rsc_21_22 = dw_img_20_25.select('label_1').neq(dw_img_20_25.select('label_2')) \
    .And(dw_img_20_25.select('label_1').eq(dw_img_20_25.select('label_3')))

pixel_count = count_pixels(rsc_21_22)
print('Changes from 2021 to 2022 that got reverted 2023:', pixel_count.get('label_1').getInfo())

rsc_22_23 = dw_img_20_25.select('label_2').neq(dw_img_20_25.select('label_3')) \
    .And(dw_img_20_25.select('label_2').eq(dw_img_20_25.select('label_4')))

pixel_count = count_pixels(rsc_22_23)
print('Changes from 2022 to 2023 that got reverted 2024:', pixel_count.get('label_2').getInfo())

rsc_23_24 = dw_img_20_25.select('label_3').neq(dw_img_20_25.select('label_4')) \
    .And(dw_img_20_25.select('label_3').eq(dw_img_20_25.select('label_5')))

pixel_count = count_pixels(rsc_23_24)
print('Changes from 2023 to 2024 that got reverted 2025:', pixel_count.get('label_3').getInfo())

Changes from 2020 to 2021 that got reverted 2022: 28416582
Changes from 2021 to 2022 that got reverted 2023: 47183936
Changes from 2022 to 2023 that got reverted 2024: 21401276
Changes from 2023 to 2024 that got reverted 2025: 22893903


### Removing Rapid Sequence Change

In [17]:
# Yearly Change Images without RSC
dw_img_change_25_24_no_rsc = dw_img_24.neq(dw_img_25).And(dw_img_23.neq(dw_img_25))
dw_img_change_24_23_no_rsc  = dw_img_23.neq(dw_img_24).And(dw_img_23.neq(dw_img_25)).And(dw_img_22.neq(dw_img_24))
dw_img_change_23_22_no_rsc  = dw_img_22.neq(dw_img_23).And(dw_img_22.neq(dw_img_24)).And(dw_img_21.neq(dw_img_23))
dw_img_change_22_21_no_rsc  = dw_img_21.neq(dw_img_22).And(dw_img_21.neq(dw_img_23)).And(dw_img_20.neq(dw_img_22))
dw_img_change_21_20_no_rsc  = dw_img_20.neq(dw_img_21).And(dw_img_20.neq(dw_img_22))

change_collection_no_rsc = ee.ImageCollection.fromImages(
    [dw_img_change_25_24_no_rsc , dw_img_change_24_23_no_rsc , dw_img_change_23_22_no_rsc , dw_img_change_22_21_no_rsc , dw_img_change_21_20_no_rsc ]
)

In [18]:
count_changes_collection(change_collection_no_rsc)

{'label': {'0': 394364032, '1': 83265438, '2': 24245451, '3': 7764327, '4': 1620884, '5': 162006, 'null': 255970350}}


In [ ]:
always_change_no_rsc = dw_img_20_25.updateMask(change_collection_no_rsc.sum().eq(5))

map_no_rsc = geemap.Map(scroll_wheel_zoom = False, basemap='ROADMAP')
map_no_rsc.add_layer(
    always_change_no_rsc,
    DW_VIZ_PARAMS,
    'Dynamic World',
)
map_no_rsc

In [ ]:
s2_col = ee.ImageCollection('COPERNICUS/S2_HARMONIZED').filter(ee.Filter.date('2024-07-01', '2024-08-01')).filterBounds(DENMARK)
sentinel_img = s2_col.mosaic()
map.add_layer(
    sentinel_img,
    S2_VIZ_PARAMS, 
    'Sentinel-2 L1C',
)
map.add_layer(
    img,
    DW_VIZ_PARAMS,
    'Dynamic World',
)
map

In [ ]:
img = dw_img_20_25.select('label_5').eq(1).And(dw_img_20_25.select('label_4').eq(5))
img = img.updateMask(img.neq(0))
map = geemap.Map(scroll_wheel_zoom = False, basemap='ROADMAP')
map.add_layer(
    img,
    {
        'min' : 0,
        'max' : 8,
        'palette' : list(DW_CLASS.values()),
        'bands' : 'label_5'
    },
    'Dynamic World',
)
map